# Graph Construction Ablation: k-NN vs Pearson (20 Runs)

**Goal.** Isolate the *edge-construction criterion* as the single experimental variable. Both variants build a graph over the **same 9 sensor channels** (nodes), with the **same 6 per-channel statistics** as node features, the **same k=4**, the **same GAT architecture, seeds and data split**. The only difference is how edges are drawn:

- **k-NN (Euclidean):** each channel connects to its 4 nearest channels in Euclidean distance over the statistical feature space (*statistical-similarity* topology).
- **Pearson:** each channel connects to its 4 channels of highest absolute Pearson correlation within the window (*temporal co-movement* topology).

Because everything except the edge metric is held constant, any performance difference is attributable solely to the graph-construction method. **Validation:** Section 4 reports the Jaccard edge overlap between the two graph sets to confirm they are genuinely distinct (not collapsed onto the same graph).

**Statistical design:** 20 independent runs per variant (seeds 42–61), 95% bootstrap CIs (2000 resamples), two-sided Mann-Whitney U test.

**Outputs:**
- `models/ablation_knn/gat_knn_run_{01..20}.pt`
- `models/ablation_pearson/gat_pearson_run_{01..20}.pt`
- `results/comparison/graph_ablation.csv` (per-run metrics)
- `results/comparison/graph_ablation_summary.csv` (mean, std, CI 95%)
- `results/comparison/graph_ablation_meta.json`

## 1. Install PyTorch Geometric (Colab)

In [5]:
import importlib
if not importlib.util.find_spec('torch_geometric'):
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric'], check=True)
    print('PyTorch Geometric installed.')
else:
    print('PyTorch Geometric already available.')

PyTorch Geometric already available.


## 2. Setup and seeds

In [ ]:
import os, sys, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import stats
from torch_geometric.loader import DataLoader as PyGDataLoader
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)

# -- Environment detection --------------------------------------------------------
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    COLAB_REPO = Path('/content/gearbox-graph-ablation-2026')
    if not COLAB_REPO.exists():
        os.system(
            'git clone https://github.com/camara0729/gearbox-graph-ablation.git '
            '/content/gearbox-graph-ablation-2026'
        )
    PROJECT_ROOT = COLAB_REPO
else:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.gat import VibrationGAT, EarlyStopping, make_graph_dataloaders, epoch_step_gat
# Both builders produce channel-node graphs (9 nodes, 6 features). Only the edge
# criterion differs: Euclidean k-NN vs Pearson correlation.
from src.preprocessing import build_knn_channel_graph, build_pearson_graph

# -- Global seeds -----------------------------------------------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Project root: {PROJECT_ROOT}')

## 3. Mount Drive and locate data/processed (Colab)

**Before running this cell**, update `DRIVE_PROCESSED_PATH` to the path of the `data/processed/` folder inside your Google Drive.  
The folder must contain: `scaler.pkl`, `split_config.json`, `X_train.npy`, `X_val.npy`, `X_test.npy`, `y_train.npy`, `y_val.npy`, `y_test.npy`.

In [7]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────
# Set this to the path of data/processed/ inside your Google Drive.
# Example: 'MyDrive/my-project/data/processed'
DRIVE_PROCESSED_PATH = 'MyDrive/anomaly_detection_comparison/data/processed'
# ─────────────────────────────────────────────────────────────────────────────

REQUIRED_PROCESSED_FILES = {
    'scaler.pkl', 'split_config.json',
    'X_train.npy', 'X_val.npy', 'X_test.npy',
    'y_train.npy', 'y_val.npy', 'y_test.npy',
}

def is_valid_processed_dir(path):
    if not path.exists() or not path.is_dir():
        return False
    return REQUIRED_PROCESSED_FILES.issubset({p.name for p in path.iterdir() if p.is_file()})

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DATA_DIR = Path('/content/drive') / DRIVE_PROCESSED_PATH
    if not is_valid_processed_dir(DATA_DIR):
        raise FileNotFoundError(
            f'Could not find a valid data/processed directory at: {DATA_DIR}\n'
            'Please update DRIVE_PROCESSED_PATH at the top of this cell.'
        )
    print(f'Drive data/processed: {DATA_DIR}')
else:
    DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
    if not is_valid_processed_dir(DATA_DIR):
        raise FileNotFoundError(f'Invalid local data directory: {DATA_DIR}')
    print(f'Local data/processed: {DATA_DIR}')

print('Files found:')
for fname in sorted(REQUIRED_PROCESSED_FILES):
    print(f'  - {fname}')

Mounted at /content/drive
Drive data/processed: /content/drive/MyDrive/anomaly_detection_comparison/data/processed
Files found:
  - X_test.npy
  - X_train.npy
  - X_val.npy
  - scaler.pkl
  - split_config.json
  - y_test.npy
  - y_train.npy
  - y_val.npy


## 4. Load arrays and build both graph sets

In [ ]:
X_train = np.load(DATA_DIR / 'X_train.npy')
X_val   = np.load(DATA_DIR / 'X_val.npy')
X_test  = np.load(DATA_DIR / 'X_test.npy')
y_train = np.load(DATA_DIR / 'y_train.npy')
y_val   = np.load(DATA_DIR / 'y_val.npy')
y_test  = np.load(DATA_DIR / 'y_test.npy')

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}    y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')

# k é idêntico para as duas variantes (única diferença = critério de aresta)
K = 4

# --- Variante k-NN (canais; arestas por distância euclidiana) --------------------
print(f'Building k-NN channel graphs (k={K})...')
graphs_knn_train = build_knn_channel_graph(X_train, k=K)
graphs_knn_val   = build_knn_channel_graph(X_val,   k=K)
graphs_knn_test  = build_knn_channel_graph(X_test,  k=K)

# --- Variante Pearson (canais; arestas por correlação) ---------------------------
print(f'Building Pearson channel graphs (k={K})...')
graphs_p_train = build_pearson_graph(X_train, k_top=K)
graphs_p_val   = build_pearson_graph(X_val,   k_top=K)
graphs_p_test  = build_pearson_graph(X_test,  k_top=K)

# Attach labels
def attach_labels(graphs, y):
    for i, g in enumerate(graphs):
        g.y = torch.tensor(int(y[i]), dtype=torch.long)

for g, y in [
    (graphs_knn_train, y_train), (graphs_knn_val, y_val), (graphs_knn_test, y_test),
    (graphs_p_train,   y_train), (graphs_p_val,   y_val), (graphs_p_test,   y_test),
]:
    attach_labels(g, y)

CLASS_NAMES = ['Normal (P1)', 'Inner Race (P2)', 'Roller Element (P3)', 'Outer Race (P4)']
for split_name, y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(f'  {split_name}: {dict(zip(range(4), np.bincount(y, minlength=4)))}')

print(f'\nSanity k-NN     | x: {graphs_knn_train[0].x.shape} | edges: {graphs_knn_train[0].edge_index.shape}')
print(f'Sanity Pearson  | x: {graphs_p_train[0].x.shape} | edges: {graphs_p_train[0].edge_index.shape}')

# --- Node features devem ser IDÊNTICOS entre as variantes (mesmos nós e features) -
same_feats = all(
    torch.equal(graphs_knn_train[i].x, graphs_p_train[i].x)
    for i in range(min(500, len(graphs_knn_train)))
)
print(f'\nNode features identical (k-NN vs Pearson): {same_feats}')

# --- Verificação: os GRAFOS são genuinamente diferentes? (Jaccard das arestas) ---
def edge_set(g):
    ei = g.edge_index.numpy()
    return set(zip(ei[0].tolist(), ei[1].tolist()))

jacc = []
for a, b in zip(graphs_knn_train, graphs_p_train):
    ea, eb = edge_set(a), edge_set(b)
    union = len(ea | eb)
    jacc.append(len(ea & eb) / union if union else 1.0)
jacc = np.array(jacc)
identical_pct = float((jacc >= 0.999).mean() * 100)
print(f'Edge Jaccard overlap (train): mean={jacc.mean():.3f}, '
      f'min={jacc.min():.3f}, max={jacc.max():.3f}')
print(f'Graphs identical (Jaccard=1.0): {identical_pct:.1f}% of windows')
if identical_pct > 99.0:
    print('WARNING: graphs are essentially identical — the comparison would be '
          'meaningless. Consider differing the node definition (e.g. window-nodes).')
else:
    print('OK: the two graph constructions are genuinely distinct.')

## 5. Hyperparameters and output directories

In [9]:
# Architecture and training
N_CLASSES      = 4
HIDDEN         = 32
HEADS          = 2
NUM_LAYERS     = 2
DROPOUT        = 0.0
BATCH_SIZE     = 64
LR             = 1e-3
MAX_EPOCHS     = 200
PATIENCE       = 5
SCHED_PATIENCE = 3

# Multi-run
N_RUNS        = 20
BASE_RUN_SEED = 42  # seeds: 42, 43, ..., 61

# Output directories
MODELS_DIR  = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'comparison'

KNN_CKPT_DIR = MODELS_DIR / 'ablation_knn'
PEA_CKPT_DIR = MODELS_DIR / 'ablation_pearson'
for d in [KNN_CKPT_DIR, PEA_CKPT_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'hidden={HIDDEN}, heads={HEADS}, layers={NUM_LAYERS}, dropout={DROPOUT}')
print(f'batch={BATCH_SIZE}, lr={LR}, max_epochs={MAX_EPOCHS}, patience={PATIENCE}')
print(f'N_RUNS={N_RUNS}, seeds={BASE_RUN_SEED}..{BASE_RUN_SEED+N_RUNS-1}')
print(f'knn checkpoints  -> {KNN_CKPT_DIR}')
print(f'pearson checkpoints -> {PEA_CKPT_DIR}')
print(f'results          -> {RESULTS_DIR}')

hidden=32, heads=2, layers=2, dropout=0.0
batch=64, lr=0.001, max_epochs=200, patience=5
N_RUNS=20, seeds=42..61
knn checkpoints  -> /content/gearbox-graph-ablation-2026/models/ablation_knn
pearson checkpoints -> /content/gearbox-graph-ablation-2026/models/ablation_pearson
results          -> /content/gearbox-graph-ablation-2026/results/comparison


## 6. Training and evaluation helpers

In [10]:
def set_all_seeds(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_variant(train_graphs, val_graphs, ckpt_path, seed):
    set_all_seeds(seed)
    n_feat = train_graphs[0].x.shape[1]
    model = VibrationGAT(
        n_feat=n_feat, n_classes=N_CLASSES,
        hidden=HIDDEN, heads=HEADS, num_layers=NUM_LAYERS, dropout=DROPOUT,
    ).to(DEVICE)

    train_dl, val_dl = make_graph_dataloaders(
        train_graphs, val_graphs, batch_size=BATCH_SIZE, seed=seed,
    )
    loss_fn   = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=SCHED_PATIENCE
    )
    early_stop = EarlyStopping(patience=PATIENCE)

    best_val_loss = float('inf')
    best_epoch    = 0

    t0 = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        tr_loss, _ = epoch_step_gat(model, train_dl, loss_fn, optimizer, train=True,  device=DEVICE)
        vl_loss, _ = epoch_step_gat(model, val_dl,   loss_fn, None,      train=False, device=DEVICE)
        scheduler.step(vl_loss)
        early_stop.step(vl_loss)
        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            best_epoch    = epoch
            torch.save(model.state_dict(), ckpt_path)
        if early_stop.should_stop:
            break

    return {
        'n_feat':        n_feat,
        'best_epoch':    best_epoch,
        'best_val_loss': best_val_loss,
        'train_seconds': time.time() - t0,
    }


def evaluate(test_graphs, ckpt_path, n_feat):
    model = VibrationGAT(
        n_feat=n_feat, n_classes=N_CLASSES,
        hidden=HIDDEN, heads=HEADS, num_layers=NUM_LAYERS, dropout=DROPOUT,
    ).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, weights_only=True, map_location=DEVICE))
    model.eval()

    loader  = PyGDataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)
    loss_fn = nn.CrossEntropyLoss()

    all_preds, all_labels, all_probs = [], [], []
    total_loss, total_n = 0.0, 0
    t0 = time.time()
    with torch.no_grad():
        for batch in loader:
            batch  = batch.to(DEVICE)
            logits = model(batch.x, batch.edge_index, batch.batch)
            total_loss += loss_fn(logits, batch.y).item() * batch.num_graphs
            total_n    += batch.num_graphs
            all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(batch.y.cpu().numpy())
    elapsed = time.time() - t0

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.concatenate(all_probs, axis=0)

    try:
        auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
    except ValueError:
        auc = float('nan')

    return {
        'test_loss':             total_loss / total_n,
        'accuracy':              accuracy_score(y_true, y_pred),
        'f1_macro':              f1_score(y_true, y_pred, average='macro'),
        'precision_macro':       precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro':          recall_score(y_true, y_pred, average='macro', zero_division=0),
        'auc_ovr_macro':         auc,
        'latency_ms_per_sample': (elapsed / total_n) * 1000.0,
    }


def bootstrap_ci(data, n_resamples=2000, confidence=0.95):
    res = stats.bootstrap(
        (np.array(data),), np.mean,
        n_resamples=n_resamples, confidence_level=confidence,
        method='percentile', random_state=42,
    )
    return float(res.confidence_interval.low), float(res.confidence_interval.high)


print('Functions defined: train_variant, evaluate, bootstrap_ci')
print(f'n_feat k-NN    : {graphs_knn_train[0].x.shape[1]}')
print(f'n_feat Pearson : {graphs_p_train[0].x.shape[1]}')

Functions defined: train_variant, evaluate, bootstrap_ci
n_feat k-NN    : 54
n_feat Pearson : 6


## 7. Training: 20 runs × 2 variants

In [11]:
knn_n_feat = graphs_knn_train[0].x.shape[1]
pea_n_feat = graphs_p_train[0].x.shape[1]

training_log = []
t_total = time.time()

for run_idx in range(1, N_RUNS + 1):
    seed = BASE_RUN_SEED + run_idx - 1
    print(f'[Run {run_idx:02d}/{N_RUNS}] seed={seed}', flush=True)

    ckpt_knn = KNN_CKPT_DIR / f'gat_knn_run_{run_idx:02d}.pt'
    r_knn = train_variant(graphs_knn_train, graphs_knn_val, ckpt_knn, seed)
    print(f'  k-NN     epoch={r_knn["best_epoch"]:3d}  val_loss={r_knn["best_val_loss"]:.4f}  {r_knn["train_seconds"]:.0f}s')

    ckpt_pea = PEA_CKPT_DIR / f'gat_pearson_run_{run_idx:02d}.pt'
    r_pea = train_variant(graphs_p_train, graphs_p_val, ckpt_pea, seed)
    print(f'  Pearson  epoch={r_pea["best_epoch"]:3d}  val_loss={r_pea["best_val_loss"]:.4f}  {r_pea["train_seconds"]:.0f}s')

    training_log.append({
        'run': run_idx, 'seed': seed,
        'knn_best_epoch':    r_knn['best_epoch'],
        'knn_best_val_loss': r_knn['best_val_loss'],
        'knn_train_s':       r_knn['train_seconds'],
        'pea_best_epoch':    r_pea['best_epoch'],
        'pea_best_val_loss': r_pea['best_val_loss'],
        'pea_train_s':       r_pea['train_seconds'],
    })

print(f'\nTotal training time: {(time.time()-t_total)/60:.1f} min')
training_df = pd.DataFrame(training_log)
print(training_df[['run', 'knn_best_val_loss', 'pea_best_val_loss']].to_string(index=False))

[Run 01/20] seed=42
  k-NN     epoch= 27  val_loss=0.0039  50s
  Pearson  epoch= 11  val_loss=0.1934  26s
[Run 02/20] seed=43
  k-NN     epoch= 20  val_loss=0.0042  37s
  Pearson  epoch= 14  val_loss=0.1930  30s
[Run 03/20] seed=44
  k-NN     epoch= 28  val_loss=0.0051  48s
  Pearson  epoch= 20  val_loss=0.3102  39s
[Run 04/20] seed=45
  k-NN     epoch=  9  val_loss=0.0060  21s
  Pearson  epoch=  9  val_loss=0.2433  22s
[Run 05/20] seed=46
  k-NN     epoch= 46  val_loss=0.0027  75s
  Pearson  epoch= 14  val_loss=0.3993  30s
[Run 06/20] seed=47
  k-NN     epoch= 20  val_loss=0.0061  37s
  Pearson  epoch= 50  val_loss=0.0603  86s
[Run 07/20] seed=48
  k-NN     epoch= 32  val_loss=0.0028  56s
  Pearson  epoch= 37  val_loss=0.1906  66s
[Run 08/20] seed=49
  k-NN     epoch= 23  val_loss=0.0024  41s
  Pearson  epoch= 21  val_loss=0.1682  40s
[Run 09/20] seed=50
  k-NN     epoch= 39  val_loss=0.0031  66s
  Pearson  epoch=  6  val_loss=0.2282  17s
[Run 10/20] seed=51
  k-NN     epoch= 76  val_

## 8. Evaluate all 40 checkpoints on the test set

In [12]:
per_run_metrics = []

for run_idx in range(1, N_RUNS + 1):
    seed = BASE_RUN_SEED + run_idx - 1

    m_knn = evaluate(graphs_knn_test, KNN_CKPT_DIR / f'gat_knn_run_{run_idx:02d}.pt', knn_n_feat)
    m_knn.update({'variant': 'k-NN', 'run': run_idx, 'seed': seed})

    m_pea = evaluate(graphs_p_test, PEA_CKPT_DIR / f'gat_pearson_run_{run_idx:02d}.pt', pea_n_feat)
    m_pea.update({'variant': 'Pearson', 'run': run_idx, 'seed': seed})

    per_run_metrics.extend([m_knn, m_pea])
    if run_idx % 5 == 0:
        print(f'  Evaluated runs 01-{run_idx:02d}')

runs_df = pd.DataFrame(per_run_metrics)
print('\nDescriptive statistics by variant:')
print(runs_df.groupby('variant')[['f1_macro', 'auc_ovr_macro', 'accuracy']].describe().round(4))

  Evaluated runs 01-05
  Evaluated runs 01-10
  Evaluated runs 01-15
  Evaluated runs 01-20

Descriptive statistics by variant:
        f1_macro                                                          \
           count    mean     std     min     25%     50%     75%     max   
variant                                                                    
Pearson     20.0  0.9026  0.0654  0.8056  0.8505  0.8846  0.9731  0.9955   
k-NN        20.0  0.9946  0.0017  0.9918  0.9934  0.9945  0.9956  0.9976   

        auc_ovr_macro          ...                 accuracy                  \
                count    mean  ...     75%     max    count    mean     std   
variant                        ...                                            
Pearson          20.0  0.9837  ...  0.9988  0.9999     20.0  0.9041  0.0642   
k-NN             20.0  0.9998  ...  0.9999  1.0000     20.0  0.9946  0.0017   

                                                 
            min     25%     50%     75%     m

## 9. Statistical analysis: 95% Bootstrap CI + Mann-Whitney U

In [13]:
f1_knn  = runs_df[runs_df['variant'] == 'k-NN']['f1_macro'].values
f1_pea  = runs_df[runs_df['variant'] == 'Pearson']['f1_macro'].values
auc_knn = runs_df[runs_df['variant'] == 'k-NN']['auc_ovr_macro'].values
auc_pea = runs_df[runs_df['variant'] == 'Pearson']['auc_ovr_macro'].values

ci_f1_knn  = bootstrap_ci(f1_knn)
ci_f1_pea  = bootstrap_ci(f1_pea)
ci_auc_knn = bootstrap_ci(auc_knn)
ci_auc_pea = bootstrap_ci(auc_pea)

# Mann-Whitney U: non-parametric, no normality assumption — appropriate for n=20
mw_f1  = stats.mannwhitneyu(f1_knn,  f1_pea,  alternative='two-sided')
mw_auc = stats.mannwhitneyu(auc_knn, auc_pea, alternative='two-sided')

print('=== 95% Bootstrap CI (n_resamples=2000, method=percentile) ===')
print(f'  k-NN    F1 : {f1_knn.mean():.4f}  CI=[{ci_f1_knn[0]:.4f}, {ci_f1_knn[1]:.4f}]  std={f1_knn.std():.5f}')
print(f'  Pearson F1 : {f1_pea.mean():.4f}  CI=[{ci_f1_pea[0]:.4f}, {ci_f1_pea[1]:.4f}]  std={f1_pea.std():.5f}')
print(f'  k-NN    AUC: {auc_knn.mean():.4f}  CI=[{ci_auc_knn[0]:.4f}, {ci_auc_knn[1]:.4f}]  std={auc_knn.std():.5f}')
print(f'  Pearson AUC: {auc_pea.mean():.4f}  CI=[{ci_auc_pea[0]:.4f}, {ci_auc_pea[1]:.4f}]  std={auc_pea.std():.5f}')
print()
print('=== Two-sided Mann-Whitney U test (alpha=0.05) ===')
for name, mw in [('F1  macro', mw_f1), ('AUC macro', mw_auc)]:
    sig = 'SIGNIFICANT' if mw.pvalue < 0.05 else 'not significant'
    print(f'  {name}: U={mw.statistic:.0f}, p={mw.pvalue:.2e}  [{sig}]')

=== 95% Bootstrap CI (n_resamples=2000, method=percentile) ===
  k-NN    F1 : 0.9946  CI=[0.9939, 0.9953]  std=0.00165
  Pearson F1 : 0.9026  CI=[0.8741, 0.9302]  std=0.06374
  k-NN    AUC: 0.9998  CI=[0.9998, 0.9999]  std=0.00010
  Pearson AUC: 0.9837  CI=[0.9774, 0.9895]  std=0.01354

=== Two-sided Mann-Whitney U test (alpha=0.05) ===
  F1  macro: U=376, p=2.06e-06  [SIGNIFICANT]
  AUC macro: U=363, p=1.10e-05  [SIGNIFICANT]


## 10. Save results

In [ ]:
# Per-run CSV
ablation_path = RESULTS_DIR / 'graph_ablation.csv'
runs_df.to_csv(ablation_path, index=False)

# Summary CSV (mean, std, CI95)
summary_rows = []
for label, f1_vals, auc_vals in [('GAT (k-NN)', f1_knn, auc_knn), ('GAT (Pearson)', f1_pea, auc_pea)]:
    ci_f1  = bootstrap_ci(f1_vals)
    ci_auc = bootstrap_ci(auc_vals)
    summary_rows.append({
        'variant':       label,
        'f1_mean':       float(f1_vals.mean()),
        'f1_std':        float(f1_vals.std()),
        'f1_ci95_low':   ci_f1[0],
        'f1_ci95_high':  ci_f1[1],
        'auc_mean':      float(auc_vals.mean()),
        'auc_std':       float(auc_vals.std()),
        'auc_ci95_low':  ci_auc[0],
        'auc_ci95_high': ci_auc[1],
    })
summary_df = pd.DataFrame(summary_rows)
summary_path = RESULTS_DIR / 'graph_ablation_summary.csv'
summary_df.to_csv(summary_path, index=False)

# Full metadata JSON
meta = {
    'n_runs': N_RUNS,
    'seed_base': BASE_RUN_SEED,
    'k': K,
    'hyperparameters': {
        'hidden': HIDDEN, 'heads': HEADS, 'num_layers': NUM_LAYERS,
        'dropout': DROPOUT, 'batch_size': BATCH_SIZE, 'lr': LR,
        'max_epochs': MAX_EPOCHS, 'patience': PATIENCE,
    },
    'graph_builders': {
        'knn':     {'nodes': 'channels', 'edge_metric': 'euclidean_knn', 'k': K, 'symmetrize': True},
        'pearson': {'nodes': 'channels', 'edge_metric': 'abs_pearson_topk', 'k': K, 'symmetrize': True},
    },
    'edge_jaccard_overlap_train': {
        'mean': float(jacc.mean()), 'min': float(jacc.min()), 'max': float(jacc.max()),
        'pct_identical': identical_pct,
    },
    'statistical_tests': {
        'method': 'Mann-Whitney U two-sided',
        'alpha': 0.05,
        'f1_macro':      {'U': float(mw_f1.statistic),  'p_value': float(mw_f1.pvalue)},
        'auc_ovr_macro': {'U': float(mw_auc.statistic), 'p_value': float(mw_auc.pvalue)},
    },
    'confidence_intervals': {
        'method': 'bootstrap percentile, n_resamples=2000, confidence=0.95, random_state=42',
        'knn':     {'f1_mean': float(f1_knn.mean()),  'f1_ci95': list(ci_f1_knn),  'auc_mean': float(auc_knn.mean()), 'auc_ci95': list(ci_auc_knn)},
        'pearson': {'f1_mean': float(f1_pea.mean()),  'f1_ci95': list(ci_f1_pea),  'auc_mean': float(auc_pea.mean()), 'auc_ci95': list(ci_auc_pea)},
    },
}
with open(RESULTS_DIR / 'graph_ablation_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f'Saved: {ablation_path}')
print(f'Saved: {summary_path}')
print(f'Saved: {RESULTS_DIR / "graph_ablation_meta.json"}')
print()
print('=== Summary for the paper ===')
for _, row in summary_df.iterrows():
    print(
        f'  {row["variant"]:14s} | '
        f'F1={row["f1_mean"]:.4f} [{row["f1_ci95_low"]:.4f},{row["f1_ci95_high"]:.4f}] | '
        f'AUC={row["auc_mean"]:.4f} [{row["auc_ci95_low"]:.4f},{row["auc_ci95_high"]:.4f}]'
    )
print(f'\n  Mann-Whitney F1:  U={mw_f1.statistic:.0f},  p={mw_f1.pvalue:.2e}')
print(f'  Mann-Whitney AUC: U={mw_auc.statistic:.0f}, p={mw_auc.pvalue:.2e}')